### Part 1

In [1]:
!pytest tests/test_loss.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_loss.py ...                                                   [100%]

============================== 3 passed in 3.30s ===============================


In [2]:
!pytest tests/test_attention.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 4 items                                                              

tests/test_attention.py ....                                             [100%]

=============================== warnings summary ===============================
tests/test_attention.py::TestAttentionCorrectness::test_backward[dtype0-0.05-0.05]
  /workspace/efdl_hw/week06_dl_arithmetic/homework/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

-- Docs: https://docs.pyte

In [3]:
!pytest tests/test_rmsnorm.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_rmsnorm.py ...                                                [100%]

============================== 3 passed in 3.12s ===============================


In [4]:
!pytest tests/test_swiglu.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 4 items                                                              

tests/test_swiglu.py ....                                                [100%]

=============================== warnings summary ===============================
tests/test_swiglu.py::TestSwiGLUCorrectness::test_backward[dtype0-5e-05-5e-05]
  /workspace/efdl_hw/week06_dl_arithmetic/homework/.venv/lib/python3.11/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

-- Docs: https://docs.pytest.o

In [5]:
!pytest tests/test_e2e.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 3 items                                                              

tests/test_e2e.py ...                                                    [100%]

============================== 3 passed in 6.20s ===============================


### Part 2

### 1-2

In [8]:
!pytest tests/test_optimizer.py

============================= test session starts ==============================
platform linux -- Python 3.11.14, pytest-9.0.2, pluggy-1.6.0
rootdir: /workspace/efdl_hw/week06_dl_arithmetic/homework
configfile: pyproject.toml
collected 1 item                                                               

tests/test_optimizer.py .                                                [100%]

============================== 1 passed in 2.93s ===============================


In [9]:
!cat efficient_optimizer/ademamix.py

import math
 
import torch
from torch import Tensor
from torch.distributed.tensor import DTensor
from torch.optim import Optimizer
from typing import Optional
 
# HINT: you may want to change these functions
def linear_warmup_scheduler(step, alpha_end, alpha_start=0, warmup=1):
    return torch.where(
        step < warmup,
        (1.0-step / float(warmup)) * alpha_start + step / float(warmup) * alpha_end,
        alpha_end
    )


def linear_hl_warmup_scheduler(step, beta_end, beta_start=0, warmup=1):
    def f(beta, eps=1e-8):
        return math.log(0.5)/torch.log(beta+eps)-1

    return torch.where(
        step < warmup,
        torch.pow(0.5, 1/(((1.0-step / float(warmup)) * f(beta_start) + step / float(warmup) * f(beta_end))+1)),
        beta_end
    )


@torch.compile(fullgraph=True) # you can comment out this line for subtask 1
def ademamix_foreach_fn(
    params: list[Tensor],
    grads: list[Tensor],
    exp_avg_fasts: list[Tensor],
    exp_avg_slows: list[Tensor],
    exp_

### 3-4

In [2]:
import os
os.environ["TORCH_LOGS"] = "+output_code"
os.environ["TORCH_LOGS_OUT"] = "compiled_kernels.log"

In [3]:
import pytest
import torch
import torch.nn as nn

from optimizer.ademamix import AdEMAMix as AdemamixForloop
from efficient_optimizer.ademamix import AdEMAMix as AdemamixForeach

import torch._dynamo as dynamo
dynamo.config.recompile_limit = 8

HIDDEN_DIM = 16
NUM_LAYERS = 3
NUM_STEPS = 100

def _build_model(device: torch.device, dtype: torch.dtype) -> nn.Module:
    torch.manual_seed(0)
    layers = [nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=True) for _ in range(NUM_LAYERS)]
    return nn.Sequential(*layers).to(device=device, dtype=dtype)


def _assert_models_close(model_a: nn.Module, model_b: nn.Module, step: int, rtol: float=1e-5, atol: float=1e-6) -> None:
    a = dict(model_a.named_parameters())
    b = dict(model_b.named_parameters())
    assert a.keys() == b.keys()

    for name in a.keys():
        pa, pb = a[name].data, b[name].data
        max_diff = (pa - pb).abs().max().item()
        assert torch.allclose(pa, pb, atol=atol, rtol=rtol), (
            f"Param mismatch at step={step}, name={name}, max_diff={max_diff}"
        )

In [4]:
def _apply_random_grads(model_a: nn.Module, model_b: nn.Module) -> None:
    torch.manual_seed(0)
    a = dict(model_a.named_parameters())
    b = dict(model_b.named_parameters())
    for name in a.keys():
        g = torch.randn_like(a[name].data)
        a[name].grad = g
        b[name].grad = g.clone()

In [5]:
device = 'cuda'
dtype = torch.float32

model_efficient = _build_model(device, dtype)

opt_efficient = AdemamixForeach(model_efficient.parameters(), lr=1e-2, weight_decay=0.1, alpha_warmup=51, beta3_warmup=51)

In [6]:
model_with_adam = _build_model(device, dtype)
opt_adam = torch.optim.Adam(model_with_adam.parameters(), lr=1e-2, weight_decay=0.1, foreach=True)

In [7]:
torch._dynamo.reset()
_apply_random_grads(model_efficient, model_with_adam)
opt_efficient.step()

V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] Output code: 
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] # AOT ID: ['1_inference']
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import torch
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import math
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import random
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import os
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] import tempfile
V0312 18:47:19.713000 102253 torch/_inductor/codecache.py:1236] [0/0] [__output_code] from math import inf, nan
V0312 18:47:19.713000 102253 torch/_inductor/codecac

In [8]:
torch._dynamo.reset()
opt_adamw = torch.optim.AdamW(model_with_adam.parameters(), lr=1e-2, weight_decay=0.1, foreach=True)
compiled_adamw_step = torch.compile(opt_adamw.step)
_apply_random_grads(model_efficient, model_with_adam)
compiled_adamw_step()

W0312 18:47:19.781000 102253 torch/_logging/_internal.py:1154] [0/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] Output code: 
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] # AOT ID: ['1_inference']
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import torch
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import math
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import random
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import os
V0312 18:47:20.036000 102253 torch/_inductor/codecache.py:1236] [1/0] [__output_code] import tempfile
V0312 18:47

если сделать поиск по "@triton.jit" то и там и там по 2

**у адама там отдельно считается керел для скаляров (например beta1) и для тензоров**

### Part 4

In [20]:
!torchrun --nproc_per_node=2 tests/bench_e2e.py

W0313 14:33:18.507000 32453 torch/distributed/run.py:774] 
W0313 14:33:18.507000 32453 torch/distributed/run.py:774] *****************************************
W0313 14:33:18.507000 32453 torch/distributed/run.py:774] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0313 14:33:18.507000 32453 torch/distributed/run.py:774] *****************************************
Running baseline training...
  Warmup run (compiling kernels)...
Running baseline training...
  Warmup run (compiling kernels)...
Training with 2 GPU(s)
Device: cuda:0
Model config: TransformerConfig(vocab_size=32000, hidden_dim=256, num_heads=32, num_layers=24, intermediate_dim=384, max_seq_len=1024, dropout=0.0, rope_theta=10000.0, rms_norm_eps=1e-06)
Model parameters: 29,765,888

Training for 1 epochs (10 steps)
Batch size: 2 x 2 = 4
Sequence length: 1024
----------

видим небольшое ускорение, и большое улучшение по памяти

In [22]:
import gc
from efficient_model.swiglu import SwiGLUFeedForward
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    layer = SwiGLUFeedForward(hidden_dim=256, intermediate_dim=384).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    output = layer(input)


    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [23]:
import numpy as np

eff_swiglu_time = np.mean(times)
eff_swiglu_peak = np.max(peaks) / 1024 / 1024

In [24]:
eff_swiglu_time, eff_swiglu_peak

(np.float64(6.981589396794637), np.float64(81.99755859375))

In [25]:
import gc
from model.swiglu import SwiGLUFeedForward
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    layer = SwiGLUFeedForward(hidden_dim=256, intermediate_dim=384).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    output = layer(input)


    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [26]:
base_swiglu_time = np.mean(times)
base_swiglu_peak = np.max(peaks) / 1024 / 1024

In [27]:
base_swiglu_time, base_swiglu_peak

(np.float64(6.538405418395996), np.float64(96.99755859375))

In [28]:
import gc
from efficient_model.norm import RMSNorm
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    layer = RMSNorm(hidden_dim=256).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    output = layer(input)


    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [29]:
eff_rmsnorm_time = np.mean(times)
eff_rmsnorm_peak = np.max(peaks) / 1024 / 1024

In [30]:
import gc
from model.norm import RMSNorm
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    layer = RMSNorm(hidden_dim=256).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    output = layer(input)


    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [31]:
base_rmsnorm_time = np.mean(times)
base_rmsnorm_peak = np.max(peaks) / 1024 / 1024

In [32]:
print(eff_rmsnorm_time, eff_rmsnorm_peak)
print(base_rmsnorm_time, base_rmsnorm_peak)

6.671983957290649 72.25732421875
6.781055927276611 74.26220703125


In [33]:
import gc
from efficient_model.loss import CrossEntropyLoss
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    lm_head_weight = torch.randn(30000, 256).cuda()
    labels = torch.randint(0, 30000, (1, 1024)).cuda()

    loss_fn = CrossEntropyLoss()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    output = loss_fn(input, lm_head_weight, labels)

    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [34]:
eff_crossentropy_time = np.mean(times)
eff_crossentropy_peak = np.max(peaks) / 1024 / 1024

In [35]:
import gc
from model.loss import cross_entropy_loss
from torch import nn
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    linear = nn.Linear(256, 30000).cuda()
    labels = torch.randint(0, 30000, (1, 1024)).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 256).cuda()
    input = linear(input)
    output = cross_entropy_loss(input, labels)

    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [36]:
base_crossentropy_time = np.mean(times)
base_crossentropy_peak = np.max(peaks) / 1024 / 1024

In [47]:
import gc
from config import TransformerConfig
from efficient_model.attention import MultiHeadAttention
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    config = TransformerConfig(
        hidden_dim=1024,
        num_heads=8,
        max_seq_len=1024,
        dropout=0.0,
        rope_theta=10000.0,
    )
    layer = MultiHeadAttention(config).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 1024).cuda()
    output = layer(input)

    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [48]:
eff_attention_time = np.mean(times)
eff_attention_peak = np.max(peaks) / 1024 / 1024

In [49]:
import gc
from config import TransformerConfig
from model.attention import MultiHeadAttention
import torch

times = []
peaks = []

for _ in range(10):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    config = TransformerConfig(
        hidden_dim=1024,
        num_heads=8,
        max_seq_len=1024,
        dropout=0.0,
        rope_theta=10000.0,
    )
    layer = MultiHeadAttention(config).cuda()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()

    input = torch.randn(1, 1024, 1024).cuda()
    output = layer(input)

    end.record()
    torch.cuda.synchronize()

    total_ms = start.elapsed_time(end)
    peak_memory = torch.cuda.max_memory_allocated()

    gc.collect()
    torch.cuda.empty_cache()

    if _ > 3:
        times.append(total_ms)
        peaks.append(peak_memory)

In [50]:
base_attention_time = np.mean(times)
base_attention_peak = np.max(peaks) / 1024 / 1024

In [51]:
import pandas as pd

df = pd.DataFrame({
    'Part': ['SwiGLU', 'RMSNorm', 'CrossEntropy', 'Attention'],
    'Efficient': [eff_swiglu_time, eff_rmsnorm_time, eff_crossentropy_time, eff_attention_time],
    'Base': [base_swiglu_time, base_rmsnorm_time, base_crossentropy_time, base_attention_time],
    'Efficient Peak': [eff_swiglu_peak, eff_rmsnorm_peak, eff_crossentropy_peak, eff_attention_peak],
    'Base Peak': [base_swiglu_peak, base_rmsnorm_peak, base_crossentropy_peak, base_attention_peak],
})
df

,Part,Efficient,Base,Efficient Peak,Base Peak
0,SwiGLU,6.981589,6.538405,81.997559,96.997559
1,RMSNorm,6.671984,6.781056,72.257324,74.262207
2,CrossEntropy,18.235248,6.835680,99.249023,454.373047
3,Attention,24.812026,25.021755,158.279785,240.248047


CE медленее но сильно лучше по памяти (ну там по чанкам же в лингер делают поэтому логично), остальное лучше по пямяти и чуть лучше по времени